# PrivacifyDoc - Named Entity Recognition (NER)

## Table of Contents:
1. **[Data Exploration](#data-exploration)**
2. **[Data Quality Analysis](#data-quality)**
3. **[Quick Training](#quick-training)**
4. **[Training Monitoring](#training-monitoring)**
5. **[Model Testing](#model-testing)**
6. **[Optimization](#optimization)**

---

### **Project Goal:**
Train a NER model to recognize Polish entities:
- **PERSON** - names and surnames
- **ADDRESS** - addresses  
- **EMAIL** - email addresses
- **PHONE** - phone numbers
- **PESEL** - PESEL numbers
- **NIP** - NIP numbers
- **REGON** - REGON numbers

---

In [ ]:
# PRIVACIFYDOC - DATA ANALYSIS AND NER MODEL TRAINING
# Notebook for data exploration and training entity recognition models for Polish language

import sys
import os
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Add path to project modules
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

print(f"Project directory: {project_root}")
print(f"Data directory: {project_root / 'data' / 'processed'}")
print(f"Experiments directory: {project_root / 'experiments'}")

# Visualization settings
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Environment configured!")

## 1. Data Exploration <a id="data-exploration"></a>

### Let's check what we have in our dataset...

In [ ]:
# Loading training data
def load_jsonl(file_path):
    """Loads data from JSONL file."""
    data = []
    if Path(file_path).exists():
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data.append(json.loads(line))
    return data

# File paths
data_dir = project_root / 'data' / 'processed'
train_path = data_dir / 'train.jsonl'
val_path = data_dir / 'val.jsonl'
test_path = data_dir / 'test.jsonl'

# Load data
print("Loading data...")
train_data = load_jsonl(train_path)
val_data = load_jsonl(val_path)
test_data = load_jsonl(test_path)

# Basic statistics
print(f"\nDATASET SIZES:")
print(f"   Training:   {len(train_data):,} examples")
print(f"   Validation: {len(val_data):,} examples")
print(f"   Test:       {len(test_data):,} examples")
print(f"   TOTAL:      {len(train_data) + len(val_data) + len(test_data):,} examples")

if not train_data:
    print("ERROR: No training data found!")
    print(f"   Check if file exists: {train_path}")
else:
    print("Data loaded successfully!")

In [ ]:
# Entity type analysis in training set
if train_data:
    entity_counts = Counter()
    text_lengths = []
    entities_per_example = []
    
    for example in train_data:
        text = example['text']
        entities = example['entities']
        
        text_lengths.append(len(text))
        entities_per_example.append(len(entities))
        
        for start, end, entity_type in entities:
            entity_counts[entity_type] += 1
    
    print("\nENTITY TYPE DISTRIBUTION:")
    for entity_type, count in entity_counts.most_common():
        percentage = (count / sum(entity_counts.values())) * 100
        print(f"   {entity_type:8} | {count:6,} | {percentage:5.1f}%")
    
    print(f"\nTEXT STATISTICS:")
    print(f"   Average length: {sum(text_lengths)/len(text_lengths):.0f} characters")
    print(f"   Min length:     {min(text_lengths)} characters")
    print(f"   Max length:     {max(text_lengths)} characters")
    
    print(f"\nENTITIES PER EXAMPLE:")
    print(f"   Average: {sum(entities_per_example)/len(entities_per_example):.1f} entities")
    print(f"   Min:     {min(entities_per_example)} entities")
    print(f"   Max:     {max(entities_per_example)} entities")

In [ ]:
# Entity distribution visualization
if train_data and entity_counts:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Bar chart of entity types
    entities = list(entity_counts.keys())
    counts = list(entity_counts.values())
    
    axes[0,0].bar(entities, counts, color='skyblue', alpha=0.8)
    axes[0,0].set_title('Entity Count by Type', fontsize=14, weight='bold')
    axes[0,0].set_ylabel('Number of Entities')
    axes[0,0].tick_params(axis='x', rotation=45)
    for i, v in enumerate(counts):
        axes[0,0].text(i, v + max(counts)*0.01, f'{v:,}', ha='center', va='bottom')
    
    # 2. Pie chart of percentage distribution
    axes[0,1].pie(counts, labels=entities, autopct='%1.1f%%', startangle=90)
    axes[0,1].set_title('Percentage Distribution of Entity Types', fontsize=14, weight='bold')
    
    # 3. Text length histogram
    axes[1,0].hist(text_lengths, bins=30, color='lightgreen', alpha=0.7, edgecolor='black')
    axes[1,0].set_title('Text Length Distribution', fontsize=14, weight='bold')
    axes[1,0].set_xlabel('Text Length (characters)')
    axes[1,0].set_ylabel('Number of Examples')
    axes[1,0].axvline(sum(text_lengths)/len(text_lengths), color='red', 
                     linestyle='--', label=f'Average: {sum(text_lengths)/len(text_lengths):.0f}')
    axes[1,0].legend()
    
    # 4. Entities per example histogram
    axes[1,1].hist(entities_per_example, bins=range(max(entities_per_example)+2), 
                   color='coral', alpha=0.7, edgecolor='black')
    axes[1,1].set_title('Entities per Example', fontsize=14, weight='bold')
    axes[1,1].set_xlabel('Number of Entities')
    axes[1,1].set_ylabel('Number of Examples')
    axes[1,1].axvline(sum(entities_per_example)/len(entities_per_example), color='red', 
                     linestyle='--', label=f'Average: {sum(entities_per_example)/len(entities_per_example):.1f}')
    axes[1,1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization")

In [ ]:
# Training data examples
if train_data:
    print("\nTRAINING DATA EXAMPLES:")
    print("="*80)
    
    # Show 3 random examples
    import random
    sample_indices = random.sample(range(len(train_data)), min(3, len(train_data)))
    
    for i, idx in enumerate(sample_indices, 1):
        example = train_data[idx]
        text = example['text'][:200] + '...' if len(example['text']) > 200 else example['text']
        entities = example['entities']
        
        print(f"\nEXAMPLE {i}:")
        print(f"Text: {text}")
        print(f"Entities:")
        
        for start, end, entity_type in entities:
            entity_text = example['text'][start:end]
            print(f"   • {entity_type:8} | '{entity_text}'")
        print("-" * 60)
else:
    print("No data to display")

## 2. Data Quality Analysis <a id="data-quality"></a>

### Checking if data is correct and complete...

In [ ]:
# Data quality analysis
def analyze_data_quality(data, dataset_name):
    """Analyzes data quality in the dataset."""
    issues = []
    stats = {
        'total_examples': len(data),
        'empty_texts': 0,
        'empty_entities': 0,
        'invalid_spans': 0,
        'overlapping_entities': 0,
        'unique_texts': len(set(ex['text'] for ex in data))
    }
    
    for i, example in enumerate(data):
        text = example['text']
        entities = example['entities']
        
        # Check empty texts
        if not text.strip():
            stats['empty_texts'] += 1
            issues.append(f"Example {i}: empty text")
        
        # Check missing entities
        if not entities:
            stats['empty_entities'] += 1
        
        # Check invalid entity spans
        for j, (start, end, entity_type) in enumerate(entities):
            if start < 0 or end > len(text) or start >= end:
                stats['invalid_spans'] += 1
                issues.append(f"Example {i}, entity {j}: invalid span [{start}:{end}]")
        
        # Check overlapping entities
        sorted_entities = sorted(entities, key=lambda x: x[0])
        for j in range(len(sorted_entities) - 1):
            if sorted_entities[j][1] > sorted_entities[j+1][0]:
                stats['overlapping_entities'] += 1
                issues.append(f"Example {i}: overlapping entities")
                break
    
    # Calculate statistics
    stats['duplicates'] = stats['total_examples'] - stats['unique_texts']
    stats['diversity'] = stats['unique_texts'] / stats['total_examples'] * 100 if stats['total_examples'] > 0 else 0
    
    return stats, issues[:10]  # Show only first 10 issues

# Analyze each dataset
for dataset_name, data in [('TRAIN', train_data), ('VAL', val_data), ('TEST', test_data)]:
    if data:
        print(f"\nQUALITY ANALYSIS - {dataset_name}:")
        stats, issues = analyze_data_quality(data, dataset_name)
        
        print(f"   Examples: {stats['total_examples']:,}")
        print(f"   Unique: {stats['unique_texts']:,} ({stats['diversity']:.1f}%)")
        print(f"   Duplicates: {stats['duplicates']:,}")
        print(f"   Empty texts: {stats['empty_texts']}")
        print(f"   Without entities: {stats['empty_entities']}")
        print(f"   Invalid spans: {stats['invalid_spans']}")
        print(f"   Overlapping entities: {stats['overlapping_entities']}")
        
        if issues:
            print(f"   \nISSUES (first 10):")
            for issue in issues:
                print(f"      • {issue}")
        else:
            print(f"   No data quality issues found!")
    else:
        print(f"\nNo data found for {dataset_name} dataset")

## 3. Quick Model Training <a id="quick-training"></a>

### Launch training with one button!

In [ ]:
# QUICK TRAINING - ONE BUTTON
import subprocess
import time
from datetime import datetime

def quick_train():
    """Launches quick model training."""
    print("STARTING QUICK TRAINING...")
    print(f"Start: {datetime.now().strftime('%H:%M:%S')}")
    
    # Check if data exists
    if not train_data:
        print("ERROR: No training data found!")
        print("   Run first: python scripts/generate_data.py --num-examples 10000")
        return False
    
    try:
        # Change to ml directory and run training
        os.chdir(project_root)
        
        start_time = time.time()
        
        # Run training script
        cmd = ["python", "scripts/train.py", "--config", "config/training_config.yaml"]
        print(f"Command: {' '.join(cmd)}")
        
        process = subprocess.Popen(
            cmd, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.STDOUT, 
            text=True,
            bufsize=1,
            universal_newlines=True
        )
        
        print("\nTRAINING OUTPUT:")
        print("="*60)
        
        # Read output in real time
        for line in process.stdout:
            print(line.rstrip())
        
        process.wait()
        end_time = time.time()
        
        if process.returncode == 0:
            duration = end_time - start_time
            print(f"\nTRAINING COMPLETED SUCCESSFULLY!")
            print(f"Training time: {duration/60:.1f} minutes")
            print(f"Check results in: experiments/")
            return True
        else:
            print(f"\nTRAINING FAILED (code: {process.returncode})")
            return False
            
    except Exception as e:
        print(f"ERROR DURING TRAINING: {e}")
        return False

# Training control
print("TRAINING CONTROLS:")
print("   To start training, run: quick_train()")
print("   Monitoring: check section below")
print("   To stop: Ctrl+C in terminal")

In [ ]:
# RUN TRAINING HERE!
# Uncomment the line below to start training:

# quick_train()

print("TIP: Uncomment the line above and run the cell to start training!")
print("WARNING: Training may take 10-30 minutes depending on data size")

## 4. Training Monitoring <a id="training-monitoring"></a>

### Let's check how training is going...

In [ ]:
# Training monitoring
def check_experiments():
    """Checks available experiments."""
    exp_dir = project_root / 'experiments'
    
    if not exp_dir.exists():
        print("Experiments directory does not exist")
        return []
    
    experiments = list(exp_dir.glob('herbert-ner-baseline_*'))
    experiments.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    
    print(f"FOUND EXPERIMENTS ({len(experiments)}):")
    
    for i, exp in enumerate(experiments[:5]):  # Show 5 newest
        timestamp = exp.name.split('_')[-1]
        mod_time = datetime.fromtimestamp(exp.stat().st_mtime)
        
        model_path = exp / 'model'
        model_exists = model_path.exists()
        
        status = "Completed" if model_exists else "In progress"
        print(f"  {i+1}. {exp.name} | {status} | {mod_time.strftime('%H:%M:%S')}")
    
    return experiments

def show_training_progress(experiment_path):
    """Shows training progress."""
    log_files = list(experiment_path.glob('**/*.log'))
    
    if not log_files:
        print("No log files found")
        return
    
    latest_log = max(log_files, key=lambda x: x.stat().st_mtime)
    
    print(f"LATEST LOGS ({latest_log.name}):")
    print("="*50)
    
    try:
        with open(latest_log, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            # Show last 20 lines
            for line in lines[-20:]:
                print(line.rstrip())
    except Exception as e:
        print(f"Error reading log: {e}")

# Check experiments
experiments = check_experiments()

if experiments:
    latest_exp = experiments[0]
    print(f"\nCHECKING LATEST EXPERIMENT: {latest_exp.name}")
    show_training_progress(latest_exp)
else:
    print("\nNo experiments found. Run training to create them!")

## 5. Model Testing <a id="model-testing"></a>

### Let's check how well our model works...

In [ ]:
# Model testing
def test_latest_model():
    """Tests the latest trained model."""
    experiments = check_experiments()
    
    if not experiments:
        print("No experiments available for testing")
        return
    
    latest_exp = experiments[0]
    model_path = latest_exp / 'model'
    
    if not model_path.exists():
        print(f"Model does not exist: {model_path}")
        return
    
    print(f"TESTING MODEL: {latest_exp.name}")
    print("="*50)
    
    try:
        # Run test
        os.chdir(project_root)
        cmd = ["python", "scripts/final_test.py"]
        
        process = subprocess.run(
            cmd, 
            capture_output=True, 
            text=True,
            timeout=300  # 5 minutes timeout
        )
        
        if process.returncode == 0:
            print("TEST RESULTS:")
            print(process.stdout)
        else:
            print("ERROR DURING TESTING:")
            print(process.stderr)
            
    except subprocess.TimeoutExpired:
        print("Test exceeded time limit (5 minutes)")
    except Exception as e:
        print(f"Error: {e}")

# Manual test examples
def manual_test_examples():
    """Manual tests on simple examples."""
    test_examples = [
        "Jan Kowalski lives at ul. Długa 15 in Warsaw",
        "Contact: anna.nowak@company.pl, tel. +48 123 456 789",
        "PESEL: 85030112345, NIP: 1234567890",
        "Dr hab. Piotr Wiśniewski, email: p.wisniewski@university.pl",
        "Company ABC Sp. z o.o., REGON: 123456789, Kraków"
    ]
    
    print("TEST EXAMPLES:")
    for i, example in enumerate(test_examples, 1):
        print(f"  {i}. {example}")
    
    print("\nTo test the model:")
    print("   1. Make sure the model is trained")
    print("   2. Run: test_latest_model()")
    print("   3. Or use manual test in scripts/final_test.py")

manual_test_examples()

In [ ]:
# RUN TEST HERE!
# Uncomment the line below to test the latest model:

# test_latest_model()

print("TIP: Uncomment the line above to test the model!")
print("WARNING: Make sure the model is already trained")

## 6. Optimization and Experiments <a id="optimization"></a>

### How to improve model results...

In [ ]:
# OPTIMIZATION GUIDE

print(""" 
NER MODEL OPTIMIZATION GUIDE

1. RESULTS ANALYSIS:
   • F1 Score < 0.8  → More/better training data
   • F1 Score = 1.0  → Possible overfitting, check on real data  
   • Long training time → Reduce batch_size or learning_rate
   • Fast convergence → Data too simple, add more diversity

2. DATA IMPROVEMENT:
   • Increase number of examples (20k+ recommended)
   • Add more format diversity
   • Check balance between entity types
   • Add real data (not just synthetic)

3. HYPERPARAMETERS:
   • Learning rate: 2e-5 (standard), 1e-5 (slower), 5e-5 (faster)
   • Batch size: 16 (standard), 8 (less memory), 32 (more memory)
   • Epochs: 3-5 (usually sufficient)
   • Warmup steps: 10% of training steps

4. ARCHITECTURE:
   • Herbert-base-cased (currently used) - good for Polish
   • Consider herbert-large for better results
   • Check other Polish models (allegro/klej-polemo)

5. EVALUATION:
   • Test on different document types
   • Check per-entity performance  
   • Use confusion matrix for error analysis
   • Test edge cases (long names, unusual formats)

QUICK EXPERIMENTS:
""")

def quick_experiment_configs():
    """Generates different configurations for quick experiments."""
    configs = {
        "conservative": {
            "learning_rate": 1e-5,
            "num_train_epochs": 5,
            "per_device_train_batch_size": 8
        },
        "aggressive": {
            "learning_rate": 5e-5,
            "num_train_epochs": 3,
            "per_device_train_batch_size": 32
        },
        "balanced": {
            "learning_rate": 2e-5,
            "num_train_epochs": 4,
            "per_device_train_batch_size": 16
        }
    }
    
    print("READY CONFIGURATIONS:")
    for name, config in configs.items():
        print(f"\n   {name.upper()}:")
        for key, value in config.items():
            print(f"     {key}: {value}")
    
    print("\nTo use a configuration:")
    print("   1. Modify config/training_config.yaml")
    print("   2. Run quick_train() again")
    print("   3. Compare results in experiments/")

quick_experiment_configs()

print("\nNEXT STEPS:")
print("   1. Analyze current results")
print("   2. Modify configuration if needed")
print("   3. Run new experiment")
print("   4. Compare results")
print("   5. Deploy best model!")

---

## Congratulations!

You now have a complete notebook for:
- **Data analysis** of training sets
- **Quick training** with one button
- **Monitoring** training progress
- **Testing** trained models
- **Optimizing** results

### Useful links:
- [Model configuration](../config/model_config.yaml)
- [Training configuration](../config/training_config.yaml)
- [Scripts](../scripts/)
- [Experiments](../experiments/)

### Tips:
1. **Start with small experiments** (1-5k examples)
2. **Monitor F1 score** - should grow gradually
3. **Test on real data** outside training set
4. **Keep notes** about each experiment
5. **Iterate quickly** - better many small experiments than one large

---

### Need help?
If something doesn't work:
1. Check if all dependencies are installed
2. Make sure CUDA is configured (for GPU)
3. Check logs in experiments/ directory
4. Run unit tests: `python -m pytest tests/`

**Good luck with model training!**